# How Often Do ASes Exceed Their Limits?

In [ ]:
import os
import json
import pickle
import datetime
import networkx as nx
import pandas as pd
import numpy as np
from collections import defaultdict
from multiprocessing import Pool
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
font_size = 24
scale_factor = 1.2
two_sided_font_size = font_size * scale_factor

plt.rcParams["font.size"] = font_size

ipv_color = {4: "tab:blue", 6: "tab:green"}


In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

working_dir = parameters["WORKING_DIR"]
data_dir = parameters["DATA_DIR"]
data_raw_dir = parameters["DATA_RAW_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
collectors = parameters["COLLECTORS"]
image_dir = parameters["IMAGE_DIR"]
tier1 = parameters["TIER1"]
tier1_asns = [item["asn"] for item in tier1]

# start_date = datetime.datetime.strptime(start_date, "%Y-%m-%d")
# end_date = datetime.datetime.strptime(end_date, "%Y-%m-%d")


In [ ]:
## Open stats output file (overwrites on every run)
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)
_stats = open(f"{numbers_dir}/9-Above_limit.md", "w")
_stats.write("# Stats: 9-Above_limit\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
print("Stats file opened.")


## Load data

### PeeringDB data

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_peeringdb = pd.read_pickle(filename)
print(df_peeringdb.shape)

df_peeringdb.head(2)


### Prefix announced AS


In [ ]:
filename = f"{data_dir}/processed/timeseries_prefix_announced_visibility.pkl"

with open(filename, "rb") as fd:
    announced_prefixes = pickle.load(fd)


### Selected ASNs

In [ ]:
fd = open(f"{data_dir}/processed/selected_asns.pkl", "rb")
selected_asns = pickle.load(fd)
fd.close()

len(selected_asns)


### Filter Prefix Origin, Announced and PeeringDB data to selected ASNs

In [ ]:
announced_prefixes = {
    asn: announced_prefixes[asn] for asn in selected_asns if asn in announced_prefixes
}
df_peeringdb = df_peeringdb[df_peeringdb["asn"].isin(selected_asns)].copy()


### CAIDA AS Rank data

In [ ]:
df_as_rank = pd.read_json(f"{data_raw_dir}/AS_rank/asns.jsonl", lines=True)
df_as_rank.head(2)
as_rank_dict = df_as_rank.set_index("asn")["rank"].to_dict()


## Plots

### How often do ASes exceed their limits?

In [ ]:
def check_above_limit(asn, ipv):

    if asn not in announced_prefixes:
        return None
    if ipv not in announced_prefixes[asn]:
        return None

    announced_asn_data = announced_prefixes[asn]
    announced_asn_ipv = announced_asn_data[ipv]["visibility_95"]

    df_peeringdb_asn = df_peeringdb[df_peeringdb["asn"] == asn]

    if df_peeringdb_asn.empty:
        return None

    peeringdb_asn = df_peeringdb_asn.iloc[0]

    limits_ipv = peeringdb_asn[f"limits_ipv{ipv}"]
    limits_date = peeringdb_asn["dates"]

    if limits_ipv is None:
        return None

    limits_ipv = {date: limit for date, limit in zip(limits_date, limits_ipv)}

    above_limit_dates_ipv = 0
    total_dates = 0
    if len(announced_asn_ipv) > 0:
        for date, n_prefixes in announced_asn_ipv.items():

            if date not in limits_ipv:
                continue

            limit = limits_ipv[date]
            total_dates += 1

            if n_prefixes > limit:
                above_limit_dates_ipv += 1

    if total_dates == 0:
        return None

    return above_limit_dates_ipv / total_dates


In [ ]:
above_limit_dates_ipv = {
    4: [],
    6: [],
}

for ipv in [4, 6]:
    for asn in tqdm(selected_asns):
        above_limit_asn_ipv = check_above_limit(asn, ipv)
        if above_limit_asn_ipv is not None:
            above_limit_dates_ipv[ipv].append((asn, above_limit_asn_ipv))


In [ ]:
def plot_histogram_with_cdf(ipv, values, bins, color_hist):
    fig, ax1 = plt.subplots(figsize=(8, 6))

    counts, bins = np.histogram(values, bins=bins)
    bw = 0.9 * (bins[1] - bins[0])
    ax1.bar(bins[:-1], counts, width=bw, color=color_hist, align="center", alpha=0.85)

    # arrow on never
    ax1.text(
        0.05, 0.8, rf"$\leftarrow$ Never above limit",
        horizontalalignment="left", verticalalignment="top",
        transform=ax1.transAxes, fontsize=two_sided_font_size,
    )
    ax1.bar(bins[0], counts[0], width=bw, color=color_hist, align="center", alpha=1, hatch="///")

    # arrow on always
    ax1.text(
        0.95, 0.5, rf"Always above limit $\rightarrow$",
        horizontalalignment="right", verticalalignment="top",
        transform=ax1.transAxes, fontsize=two_sided_font_size,
    )
    ax1.bar(bins[-2], counts[-1], width=bw, color=color_hist, align="center", alpha=1, hatch="\\\\\\")

    ax1.set_ylabel("Number of ASes", fontsize=two_sided_font_size)
    ax1.set_xlabel("Fraction above limit (%)", fontsize=two_sided_font_size)
    ax1.set_yscale("log")
    ax1.grid(axis="y", alpha=0.75)
    ax1.set_xlim(-1.5, 101.5)

    # eCDF on a twin right axis (grey line); keeps the original ePDF bars
    ax2 = ax1.twinx()
    ecdf = np.cumsum(counts) / counts.sum()
    ax2.plot(bins[:-1], ecdf, color="grey", lw=2.5)
    ax2.set_ylim(0, 1)
    ax2.set_yticks(np.arange(0.2, 1.01, 0.2))
    ax2.set_ylabel("eCDF", fontsize=two_sided_font_size)

    plt.savefig(
        f"{image_dir}/above_limit/limit_histogram_ipv{ipv}.pdf",
        bbox_inches="tight",
        dpi=300,
    )
    plt.savefig(
        f"{image_dir}/above_limit/limit_histogram_ipv{ipv}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()


In [ ]:
bins = np.linspace(0, 102, 51)

values_ipv4 = [round(x[1] * 100) for x in above_limit_dates_ipv[4]]
values_ipv6 = [round(x[1] * 100) for x in above_limit_dates_ipv[6]]

plot_histogram_with_cdf(4, values_ipv4, bins, "tab:blue")
plot_histogram_with_cdf(6, values_ipv6, bins, "tab:green")


In [ ]:
plt.figure(figsize=(4, 0.6))
plt.axis("off")
plt.fill_between([], [], [], color="tab:blue", label="IPv4")
plt.fill_between([], [], [], color="tab:green", label="IPv6")
plt.legend(loc="upper right", ncol=2, frameon=False, fontsize=two_sided_font_size)
plt.savefig(f"{image_dir}/above_limit/legend.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/above_limit/legend.png", bbox_inches="tight", dpi=300)
plt.show()


#### Exceedance magnitude at crossings (limit-threshold sensitivity)

How far *over* the limit do ASes go when they cross it? This defends using the declared limit (rather than a
$1.5\times$ or $2\times$ multiplier) as the exceedance threshold: if most crossings clear the limit only modestly,
a stricter multiplier would miss the majority of real exceedances. We treat each **crossing event
independently** — a below$\rightarrow$above transition contributes one point, its magnitude
$(\text{prefixes}-\text{limit})/\text{limit}$ at the crossing snapshot — rather than pooling every above-limit
8h snapshot (which would let a persistent exceeder dominate and would bake in a static-value assumption).

In [ ]:
# Over-limit magnitude at each below->above crossing, pooled across ASes (each crossing is one
# independent point). Event-based, NOT per-snapshot: a persistently-exceeding AS contributes one
# point per NEW crossing, not one per 8h window, so the distribution is not dominated by always-above
# ASes and does not treat the (time-varying) exceedance value as static. Zero/undefined limits skipped.
# We record BOTH the relative magnitude (% of the limit -> answers the 1.5x/2x threshold question)
# and the absolute magnitude (prefixes over the limit -> grounds the "just a few prefixes" sentence).
_lim_map = {
    r["asn"]: (r["dates"], r["limits_ipv4"], r["limits_ipv6"])
    for _, r in df_peeringdb.iterrows()
}


def crossing_over_limit(ipv):
    rel, absol = [], []
    for asn, data in announced_prefixes.items():
        if ipv not in data or asn not in _lim_map:
            continue
        ann = data[ipv]["visibility_95"]
        dates, l4, l6 = _lim_map[asn]
        lv = l4 if ipv == 4 else l6
        if lv is None:
            continue
        limits = dict(zip(dates, lv))
        prev_above = None
        for date in sorted(ann):
            limit = limits.get(date)
            if limit is None or limit <= 0:       # magnitude undefined for a zero/absent limit
                prev_above = None
                continue
            above = ann[date] > limit
            if above and prev_above is False:      # below/at -> above == a crossing
                over = ann[date] - limit
                absol.append(over)
                rel.append(over / limit * 100.0)
            prev_above = above
    return np.array(rel), np.array(absol)


over_rel, over_abs = {}, {}
for ipv in [4, 6]:
    over_rel[ipv], over_abs[ipv] = crossing_over_limit(ipv)

# stats: relative (defends limit vs 1.5x/2x) + absolute (grounds "a handful of prefixes over")
_stats.write("## Over-limit Magnitude at Crossings (all below->above transitions)\n\n")
_stats.write("(One independent point per crossing; zero/undefined limits skipped.)\n\n")
_stats.write("### Relative (% over limit)\n\n")
_stats.write("| IPv | crossings | median | mean | <=+50% | <=+100% | >+100% |\n")
_stats.write("|-----|-----------|--------|------|--------|---------|--------|\n")
for ipv in [4, 6]:
    m = over_rel[ipv]
    _stats.write(
        f"| IPv{ipv} | {len(m):,} | +{np.median(m):.0f}% | +{np.mean(m):.0f}% | "
        f"{(m <= 50).mean() * 100:.1f}% | {(m <= 100).mean() * 100:.1f}% | {(m > 100).mean() * 100:.1f}% |\n"
    )
_stats.write("\n### Absolute (prefixes over limit)\n\n")
_stats.write("| IPv | median | mean | ==1 prefix | <=2 | <=5 | <=10 |\n")
_stats.write("|-----|--------|------|-----------|-----|-----|------|\n")
for ipv in [4, 6]:
    a = over_abs[ipv]
    _stats.write(
        f"| IPv{ipv} | {np.median(a):.0f} | {np.mean(a):.1f} | {(a == 1).mean() * 100:.1f}% | "
        f"{(a <= 2).mean() * 100:.1f}% | {(a <= 5).mean() * 100:.1f}% | {(a <= 10).mean() * 100:.1f}% |\n"
    )
_stats.write("\n")
_stats.flush()
for ipv in [4, 6]:
    m, a = over_rel[ipv], over_abs[ipv]
    print(
        f"IPv{ipv}: {len(m):,} crossings | REL median +{np.median(m):.0f}% (<=+50%: {(m<=50).mean()*100:.1f}%, "
        f"<=+100%: {(m<=100).mean()*100:.1f}%) | ABS median {np.median(a):.0f} (==1: {(a==1).mean()*100:.1f}%, "
        f"<=10: {(a<=10).mean()*100:.1f}%)"
    )

# CDF of the RELATIVE magnitude (answers 38D's limit vs 1.5x/2x). Uses the shared IPv legend
# (images/above_limit/legend.pdf) like the other figures, so no in-plot legend here.
from matplotlib.ticker import LogLocator, NullLocator
fig, ax = plt.subplots(figsize=(12, 6))          # wider, matching Fig 3 (temporal median)
for ipv in [4, 6]:
    m = np.sort(over_rel[ipv])
    ys = np.arange(1, len(m) + 1) / len(m)
    ax.step(m, ys, where="post", color=ipv_color[ipv], lw=3)
for x, ls in [(50, "--"), (100, ":")]:
    ax.axvline(x, color="grey", ls=ls, lw=1.5)
    ax.text(x, 0.02, f"+{x}%", color="grey", fontsize=16, rotation=90, ha="right", va="bottom")
ax.set_xscale("log")
ax.set_xlim(1, None)
ax.set_ylim(0, 1)
ax.set_xlabel("Over-limit magnitude (%)")
ax.set_ylabel("CDF of exceedance events")
# simpler grid: vertical lines only, at 10^k and 5*10^k
ax.xaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0, 5.0)))
ax.xaxis.set_minor_locator(NullLocator())
ax.grid(False)
ax.grid(axis="x", which="major", alpha=0.3)
plt.savefig(f"{image_dir}/above_limit/exceedance_magnitude_cdf.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/above_limit/exceedance_magnitude_cdf.png", bbox_inches="tight", dpi=300)
plt.show()

#### Breakdown by Info type

In [ ]:
info_types = df_peeringdb["info_type"].value_counts().to_dict()
print(info_types.keys())


In [ ]:
asn_below_limit_ipv4 = [x[0] for x in above_limit_dates_ipv[4] if x[1] == 0]
asn_sometimes_above_limit_ipv4 = [
    x[0] for x in above_limit_dates_ipv[4] if x[1] > 0 and x[1] < 1
]
asn_above_limit_ipv4 = [x[0] for x in above_limit_dates_ipv[4] if x[1] == 1]

asn_below_limit_ipv6 = [x[0] for x in above_limit_dates_ipv[6] if x[1] == 0]
asn_sometimes_above_limit_ipv6 = [
    x[0] for x in above_limit_dates_ipv[6] if x[1] > 0 and x[1] < 1
]
asn_above_limit_ipv6 = [x[0] for x in above_limit_dates_ipv[6] if x[1] == 1]


asn_limit_classification = {
    4: {
        "below_limit": asn_below_limit_ipv4,
        "sometimes_above_limit": asn_sometimes_above_limit_ipv4,
        "above_limit": asn_above_limit_ipv4,
    },
    6: {
        "below_limit": asn_below_limit_ipv6,
        "sometimes_above_limit": asn_sometimes_above_limit_ipv6,
        "above_limit": asn_above_limit_ipv6,
    },
}


In [ ]:
for ipv in [4, 6]:
    print(f"IPv{ipv}:")
    for classification in ["below_limit", "sometimes_above_limit", "above_limit"]:
        n_asns_classification = len(asn_limit_classification[ipv][classification])
        n_asns_total = len(above_limit_dates_ipv[ipv])
        percentage = n_asns_classification / n_asns_total * 100
        print(f"{classification}: {n_asns_classification} ASes ({percentage:.2f}%)")


In [ ]:
_stats.write("## Exceedance Classification\n\n")
for ipv in [4, 6]:
    n_total = len(above_limit_dates_ipv[ipv])
    n_never = len(asn_limit_classification[ipv]["below_limit"])
    n_sometime = len(asn_limit_classification[ipv]["sometimes_above_limit"])
    n_always = len(asn_limit_classification[ipv]["above_limit"])
    _stats.write(f"### IPv{ipv} (total ASes with non-zero limit: {n_total:,})\n\n")
    _stats.write(f"- Never exceeded:      {n_never:,} ({n_never/n_total*100:.2f}%)\n")
    _stats.write(
        f"- Sometimes exceeded:  {n_sometime:,} ({n_sometime/n_total*100:.2f}%)\n"
    )
    _stats.write(
        f"- Always exceeded:     {n_always:,} ({n_always/n_total*100:.2f}%)\n\n"
    )
_stats.flush()


In [ ]:
latex_type_map = {
    "Cable/DSL/ISP": "Cable/DSL/ISP",
    "N/A": "N/A",
    "NSP": "NSP",
    "Content": "Content",
    "Educational/Research": "Edu/Research",
    "Enterprise": "Enterprise",
    "Network Services": "Network Services",
    "Non-Profit": "Non-Profit",
    "Government": "Government",
    "Route Server": "Route Server",
    "Route Collector": "Route Collector",
}

class_order = ["below_limit", "sometimes_above_limit", "above_limit"]
ipv_order = [4, 6]


def pct(n, d):
    return 0.0 if d == 0 else (100.0 * n / d)


# Preserve the actual PeeringDB type list
info_type_order = list(info_types.keys())

# Precompute percentages per type so we can sort rows
# cells index mapping:
# 0 Never IPv4, 1 Never IPv6, 2 Partially IPv4, 3 Partially IPv6, 4 Always IPv4, 5 Always IPv6
type_to_cells = {}
for info_type in info_type_order:
    cells = []
    for cls in class_order:
        for ipv in ipv_order:
            asns = set(asn_limit_classification[ipv][cls])
            count_in_class = df_peeringdb[
                df_peeringdb["asn"].isin(asns)
                & (df_peeringdb["info_type"] == info_type)
            ].shape[0]
            total = info_types.get(info_type, 0)
            cells.append(pct(count_in_class, total))
    type_to_cells[info_type] = cells

# Sort by Partially percentage (descending): IPv4 first, then IPv6
sorted_info_types = sorted(
    info_type_order,
    key=lambda t: (type_to_cells[t][2], type_to_cells[t][3]),
    reverse=True,
)

print(r"\begin{table}[t]")
print(r"\centering")
print(
    r"\caption{Exceedance rates by network type. Remaining ASes never exceeded their declared limit.}"
)
print(r"\label{tab:exceedance-by-type}")
print(r"\begin{tabular}{@{}lcccc@{}}")
print(r"\toprule")
print(
    r"\textbf{Type} & \multicolumn{2}{c}{\textbf{Partially}} & \multicolumn{2}{c}{\textbf{Always}} \\"
)
print(r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}")
print(r"& \textbf{IPv4} & \textbf{IPv6} & \textbf{IPv4} & \textbf{IPv6} \\")
print(r"\midrule")

for info_type in sorted_info_types:
    row_label = latex_type_map.get(info_type, info_type if info_type else "Unknown")
    cells_pct = type_to_cells[info_type]
    # indices 2,3 = Partially IPv4/IPv6; 4,5 = Always IPv4/IPv6
    partial_ipv4 = cells_pct[2]
    partial_ipv6 = cells_pct[3]
    always_ipv4 = cells_pct[4]
    always_ipv6 = cells_pct[5]

    # blue: Cable/DSL/ISP Partially IPv4 — highest episodic exceedance rate
    if info_type == "Cable/DSL/ISP":
        partial_ipv4_str = (
            r"\cellcolor{orange!15}\textbf{" + f"{partial_ipv4:.1f}\\%" + r"}"
        )
    else:
        partial_ipv4_str = f"{partial_ipv4:.1f}\\%"

    # orange: Network Services Always IPv4 — highest persistent-misconfiguration rate
    if info_type == "Network Services":
        always_ipv4_str = (
            r"\cellcolor{orange!15}\textbf{" + f"{always_ipv4:.1f}\\%" + r"}"
        )
    else:
        always_ipv4_str = f"{always_ipv4:.1f}\\%"

    row = f"{row_label} & {partial_ipv4_str} & {partial_ipv6:.1f}\\% & {always_ipv4_str} & {always_ipv6:.1f}\\% \\\\"
    print(row)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


In [ ]:
_stats.write("## Exceedance by Network Type\n\n")
_stats.write("| Type | Partially IPv4 | Partially IPv6 | Always IPv4 | Always IPv6 |\n")
_stats.write("|------|---------------|---------------|-------------|-------------|\n")
for info_type in sorted_info_types:
    row_label = latex_type_map.get(info_type, info_type if info_type else "Unknown")
    cells_pct = type_to_cells[info_type]
    partial_ipv4 = cells_pct[2]
    partial_ipv6 = cells_pct[3]
    always_ipv4 = cells_pct[4]
    always_ipv6 = cells_pct[5]
    _stats.write(
        f"| {row_label} | {partial_ipv4:.1f}% | {partial_ipv6:.1f}% | {always_ipv4:.1f}% | {always_ipv6:.1f}% |\n"
    )
_stats.write("\n")
_stats.flush()
print("Exceedance by network type written to stats file.")


#### Breakdown by AS Rank

In [ ]:
def classify_rank(asn):
    if asn not in as_rank_dict:
        return -1, "Unknown"
    rank = as_rank_dict.get(asn)

    if asn in tier1_asns:
        return rank, "Tier-1"
    if rank <= 100:
        return rank, "Major"
    if rank <= 1000:
        return rank, "Regional"
    return rank, "Peripheral"


In [ ]:
# compute how many ASes in each rank class are above the limit for IPv4 and IPv6 in our dataset
how_many_asns_per_rank_class = {4: {}, 6: {}}
for ipv in [4, 6]:
    for asn, _ in above_limit_dates_ipv[ipv]:
        _, rank_class = classify_rank(asn)
        if rank_class == "Unknown":
            continue
        if rank_class not in how_many_asns_per_rank_class[ipv]:
            how_many_asns_per_rank_class[ipv][rank_class] = 0
        how_many_asns_per_rank_class[ipv][rank_class] += 1

rank_classes = ["Tier-1", "Major", "Regional", "Peripheral"]
rank_classes_color = {
    "Tier-1": "tab:red",
    "Major": "tab:cyan",
    "Regional": "tab:purple",
    "Peripheral": "tab:olive",
}


In [ ]:
for ipv in asn_limit_classification:

    plt.figure(figsize=(8, 6))

    for classification_index, classification in enumerate(
        ["sometimes_above_limit", "above_limit"]
    ):

        ipv_classification_asns = asn_limit_classification[ipv][classification]
        ranks = [
            as_rank_dict[asn] for asn in ipv_classification_asns if asn in as_rank_dict
        ]
        ranks_classified = [classify_rank(asn)[1] for asn in ipv_classification_asns]
        ranks_classified = [r for r in ranks_classified if r != "Unknown"]

        for rank_class_index, rank_class in enumerate(rank_classes):
            n_in_class = sum(1 for r in ranks_classified if r == rank_class)
            total_in_class = how_many_asns_per_rank_class[ipv][rank_class]

            percentage_in_class = n_in_class / total_in_class * 100
            print(
                f"IPv{ipv} - {classification} - {rank_class}: {n_in_class}/{total_in_class} ASes ({percentage_in_class:.2f}%)"
            )

            plt.bar(
                classification_index + rank_class_index * 0.2,
                percentage_in_class,
                width=0.2,
                color=rank_classes_color[rank_class],
                label=rank_class if classification_index == 0 else "",
            )

            if percentage_in_class > 0:
                plt.text(
                    classification_index + rank_class_index * 0.2,
                    percentage_in_class + 0.3,
                    f"{percentage_in_class:.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=18,
                )

    plt.xticks(
        [i + 0.3 for i in range(2)],
        ["Partially", "Always"],
        fontsize=two_sided_font_size,
    )
    plt.ylim(0, 17)
    # plt.xlabel("Exceedance Classification")
    plt.ylabel("Percentage of ASes", fontsize=two_sided_font_size)

    plt.grid(axis="y", alpha=0.25)
    plt.savefig(
        f"{image_dir}/above_limit/as_rank_distribution_ipv{ipv}.pdf",
        bbox_inches="tight",
        dpi=300,
    )
    plt.savefig(
        f"{image_dir}/above_limit/as_rank_distribution_ipv{ipv}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()


In [ ]:
_stats.write("## Exceedance by AS Rank Class\n\n")
for ipv in [4, 6]:
    _stats.write(f"### IPv{ipv}\n\n")
    for classification in ["sometimes_above_limit", "above_limit"]:
        label = "Partially" if classification == "sometimes_above_limit" else "Always"
        _stats.write(f"**{label}:**\n\n")
        ipv_classification_asns = asn_limit_classification[ipv][classification]
        ranks_classified = [classify_rank(asn)[1] for asn in ipv_classification_asns]
        ranks_classified = [r for r in ranks_classified if r != "Unknown"]
        for rank_class in rank_classes:
            n_in_class = sum(1 for r in ranks_classified if r == rank_class)
            total_in_class = how_many_asns_per_rank_class[ipv].get(rank_class, 0)
            pct = n_in_class / total_in_class * 100 if total_in_class > 0 else 0.0
            _stats.write(
                f"- {rank_class}: {n_in_class}/{total_in_class} ({pct:.1f}%)\n"
            )
        _stats.write("\n")
_stats.flush()


In [ ]:
plt.figure(figsize=(4, 0.6))
for rank_class in rank_classes_color:
    if rank_class == "Tier-1":
        continue

    plt.bar(
        [0],
        [0],
        color=rank_classes_color[rank_class],
        label=rank_class,
    )

plt.legend(loc="upper right", ncol=3, frameon=False, fontsize=two_sided_font_size)
plt.axis("off")
plt.savefig(
    f"{image_dir}/above_limit/legend_rank_class.pdf", bbox_inches="tight", dpi=300
)
plt.savefig(
    f"{image_dir}/above_limit/legend_rank_class.png", bbox_inches="tight", dpi=300
)
plt.show()


In [ ]:
_stats.close()
print(f"Stats written to {numbers_dir}/9-Above_limit.md")
